# 11 — V5: Adding Review Score
## Customer Analytics Platform

Purpose: Test whether adding one richer feature (average review score)
improves on V4's weak result (ROC AUC 0.5461, 180-day window).

Important: a review can only exist after an order is delivered, often
weeks after purchase. Filtering by order date alone is NOT enough to
avoid leakage here - a review for a pre-cutoff order could still be
WRITTEN after the cutoff. This notebook filters by review_creation_date,
not order_purchase_timestamp, to keep the same features-before-cutoff
rule that fixed the leakage in 10_Temporal_CLV_Target.

In [0]:
# install xgboost - each notebook has its own python environment

%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
# load bronze customers (for clean mapping) + silver orders, payments, reviews

bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"
silver_path = "/Volumes/workspace/default/olist_raw_data/silver"

customers_bronze_df = spark.read.format("delta").load(f"{bronze_path}/customers")
orders_df           = spark.read.format("delta").load(f"{silver_path}/orders")
payments_df         = spark.read.format("delta").load(f"{silver_path}/payments")
reviews_df          = spark.read.format("delta").load(f"{silver_path}/reviews")

print("tables loaded")

In [0]:
# confirm review table has the columns we need

reviews_df.printSchema()
reviews_df.show(3)

# silver reviews is missing review_creation_date - check if bronze still has it
# same pattern as the customers_df issue in notebook 10

reviews_bronze_df = spark.read.format("delta").load(f"{bronze_path}/reviews")

reviews_bronze_df.printSchema()
reviews_bronze_df.show(3)

In [0]:
# same join as 10_Temporal_CLV_Target - bronze customers, filter nulls

from pyspark.sql.functions import col

orders_payments_customers = orders_df \
    .join(payments_df, "order_id", "left") \
    .join(customers_bronze_df.select("customer_id", "customer_unique_id"), "customer_id", "left") \
    .select(
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp",
        "total_payment_value"
    ) \
    .filter(col("customer_unique_id").isNotNull())

print(f"orders joined : {orders_payments_customers.count():,}")

In [0]:
feature_cutoff_date = "2018-01-01"
label_window_end    = "2018-07-01"   # 180 days

print(f"feature window : orders before {feature_cutoff_date}")
print(f"label window   : orders from {feature_cutoff_date} to {label_window_end}")

In [0]:
from pyspark.sql.functions import (
    max as spark_max, count, sum as spark_sum, round as spark_round,
    datediff, lit, to_date
)

historical_orders = orders_payments_customers.filter(
    col("order_purchase_timestamp") < feature_cutoff_date
)

rfm_historical = historical_orders \
    .groupBy("customer_unique_id") \
    .agg(
        datediff(
            to_date(lit(feature_cutoff_date)),
            spark_max("order_purchase_timestamp")
        ).alias("recency"),
        count("order_id").alias("frequency"),
        spark_round(spark_sum("total_payment_value"), 2).alias("monetary")
    ) \
    .fillna({"monetary": 0.0})

print(f"customers with history before cutoff : {rfm_historical.count():,}")

In [0]:
# combine bronze reviews (has the date) with silver reviews (has the cleaned score)
# joined on order_id, since neither table alone has both pieces we need

reviews_bronze_df = spark.read.format("delta").load(f"{bronze_path}/reviews")

reviews_combined = reviews_df.select("order_id", "review_score") \
    .join(
        reviews_bronze_df.select("order_id", "review_creation_date"),
        "order_id",
        "inner"
    )

# filter by the actual review timing, not the order timing
# a review written after the cutoff wasn't knowable at the cutoff,
# even if the order itself happened earlier

reviews_known_at_cutoff = reviews_combined.filter(
    col("review_creation_date") < feature_cutoff_date
)

print(f"reviews known at cutoff : {reviews_known_at_cutoff.count():,}")
reviews_known_at_cutoff.show(5)

In [0]:
# average review score per customer, using only reviews knowable at the cutoff

from pyspark.sql.functions import avg as spark_avg

review_score_by_customer = orders_payments_customers \
    .join(reviews_known_at_cutoff.select("order_id", "review_score"), "order_id", "inner") \
    .groupBy("customer_unique_id") \
    .agg(spark_round(spark_avg("review_score"), 2).alias("avg_review_score"))

print(f"customers with a known review score before cutoff : {review_score_by_customer.count():,}")
review_score_by_customer.show(5)

In [0]:
# add review score to features - fillna with overall average (neutral default,
# not 0, which would look like a terrible review instead of "unknown")

overall_avg_review = reviews_known_at_cutoff.agg(
    spark_avg("review_score")
).collect()[0][0]

print(f"overall average review score (fill value) : {overall_avg_review:.2f}")

rfm_historical_v5 = rfm_historical \
    .join(review_score_by_customer, "customer_unique_id", "left") \
    .fillna({"avg_review_score": round(overall_avg_review, 2)})

print(f"final feature set : {rfm_historical_v5.count():,} customers")
rfm_historical_v5.show(5)

In [0]:
# future orders / target - identical logic to 10_Temporal_CLV_Target

future_orders = orders_payments_customers.filter(
    (col("order_purchase_timestamp") >= feature_cutoff_date) &
    (col("order_purchase_timestamp") < label_window_end)
)

future_activity = future_orders \
    .groupBy("customer_unique_id") \
    .agg(count("order_id").alias("future_order_count"))

print(f"customers with future activity : {future_activity.count():,}")

In [0]:
# combine features + target

modeling_df = rfm_historical_v5 \
    .join(future_activity, "customer_unique_id", "left") \
    .fillna({"future_order_count": 0}) \
    .withColumn(
        "is_repeat_customer_90d",
        (col("future_order_count") > 0).cast("int")
    )

print(f"final modeling set : {modeling_df.count():,} customers")
modeling_df.groupBy("is_repeat_customer_90d").count().show()

In [0]:
# save as its own gold table - keeps v4 and v5 both queryable for comparison

gold_path = "/Volumes/workspace/default/olist_raw_data/gold"

modeling_df.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/rfm_temporal_features_v5")

print("v5 features saved")

In [0]:
# convert to pandas, train/test split

import pandas as pd
from sklearn.model_selection import train_test_split

pdf = modeling_df.select(
    "recency", "frequency", "monetary", "avg_review_score", "is_repeat_customer_90d"
).toPandas()

pdf = pdf.fillna(0)

X = pdf[["recency", "frequency", "monetary", "avg_review_score"]]
y = pdf["is_repeat_customer_90d"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"train : {X_train.shape[0]} rows")
print(f"test  : {X_test.shape[0]} rows")
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

In [0]:
# reuse the same tuned params from 09 - keeps this a fair, controlled comparison to v4

import mlflow

mlflow.set_experiment(
    "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"
)

runs_v3 = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'xgboost_clv_v3_tuned'",
    order_by=["metrics.roc_auc DESC"]
)

tuned_params = {
    "max_depth":        int(runs_v3.iloc[0]["params.max_depth"]),
    "learning_rate":    float(runs_v3.iloc[0]["params.learning_rate"]),
    "n_estimators":     int(runs_v3.iloc[0]["params.n_estimators"]),
    "subsample":        float(runs_v3.iloc[0]["params.subsample"]),
    "colsample_bytree": float(runs_v3.iloc[0]["params.colsample_bytree"]),
    "min_child_weight": int(runs_v3.iloc[0]["params.min_child_weight"]),
    "gamma":            float(runs_v3.iloc[0]["params.gamma"])
}

print(tuned_params)

In [0]:
# train v5, log to mlflow, compare directly to v4's 0.5461

import xgboost as xgb
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

with mlflow.start_run(run_name="xgboost_clv_v5_review_score"):

    model_v5 = xgb.XGBClassifier(
        **tuned_params,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    model_v5.fit(X_train, y_train)

    y_pred      = model_v5.predict(X_test)
    y_pred_prob = model_v5.predict_proba(X_test)[:, 1]

    roc_auc   = roc_auc_score(y_test, y_pred_prob)
    avg_prec  = average_precision_score(y_test, y_pred_prob)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)

    mlflow.log_params(tuned_params)
    mlflow.log_param("features", "recency, frequency, monetary, avg_review_score")
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("average_precision", avg_prec)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.xgboost.log_model(model_v5, "xgboost_clv_v5_review_score")

    print("V5 Complete")
    print("-" * 40)
    print(f"roc auc             : {roc_auc:.4f}   (V4 was 0.5461)")
    print(f"average precision   : {avg_prec:.4f}   (V4 was 0.0286)")
    print(f"precision           : {precision:.4f}")
    print(f"recall              : {recall:.4f}")
    print("-" * 40)

In [0]:
# feature importance - is review_score pulling real weight?

import matplotlib.pyplot as plt

feature_names = ["recency", "frequency", "monetary", "avg_review_score"]
importance    = model_v5.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importance)
plt.xlabel("importance score")
plt.title("XGBoost Feature Importance — V5 (with review score)")
plt.tight_layout()
plt.show()

for name, score in zip(feature_names, importance):
    print(f"  {name:<18} : {score:.4f}")

## V5 Summary — Review Score

Tested whether adding average review score (computed only from reviews
knowable before the cutoff date) improves on V4's RFM-only result.

| Metric | V4 (RFM only) | V5 (+ review score) |
|--------|--------------:|---------------------:|
| ROC AUC | 0.5461 | 0.5345 |
| Average Precision | 0.0286 | 0.0366 |

Result: mixed, not a clear win. ROC AUC slightly decreased, Average
Precision slightly increased. With only 543 positive examples in the
full dataset, a single train/test split isn't large enough to reliably
distinguish a real effect from sampling noise at this scale.

Conclusion: one feature at a time, on a single split, isn't a rigorous
enough test at this sample size. Next iteration (V6) will combine
multiple features (review score, delivery experience, product category)
in one test, and use cross-validation instead of a single split, to
get a result that's actually trustworthy either way.

Feature importance did show avg_review_score contributing meaningfully
(16.6%) rather than being ignored by the model - encouraging, even
though it didn't clearly move the headline metric on its own.

In [0]:
# check what's actually available for delivery + category features

order_items_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/olist_raw_data/silver/order_items"
)
products_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/olist_raw_data/silver/products"
)

print("ORDERS schema (checking for delivery_days, is_late_delivery):")
orders_df.printSchema()

print("\nORDER_ITEMS schema (checking for product_id link):")
order_items_df.printSchema()

print("\nPRODUCTS schema (checking for category column):")
products_df.printSchema()